# TSE — Tribunal Superior Eleitoral

Jurisprudence search (`cjsg`) for the Justiça Eleitoral, via the
[Pesquisa de Jurisprudência](https://jurisprudencia.tse.jus.br/#/jurisprudencia/pesquisa)
SPA.

**This is the only juscraper scraper backed by browser automation**
(Playwright) instead of `requests`: every search request requires an
hCaptcha token minted by the frontend itself, which a plain HTTP client
cannot produce. `TSEScraper` drives a real (headless by default) browser
to let the SPA build and submit its own search, then captures the JSON
response — see [`docs/captcha/tse_captcha.md`](https://github.com/jtrecenti/juscraper/blob/main/docs/captcha/tse_captcha.md)
for the full writeup and why this is an explicit exception to juscraper's
usual "requests only" scrapers.

Requires the optional `tse` extra:

```bash
pip install -e ".[tse]"
playwright install chromium
```

| Feature | Available |
|---------|-----------|
| cjsg    | Yes       |
| cjpg    | No        |
| cpopg   | No        |
| cposg   | No        |


## Basic search

In [1]:
import juscraper as jus

tse = jus.scraper("tse")
df = tse.cjsg("direito de resposta", paginas=1, tamanho_pagina=10)

print(df.shape)
df[["processo", "classe", "relator", "data_julgamento"]].head(3)

(10, 14)


,processo,classe,relator,data_julgamento
0,06017689220266000000,DIREITO DE RESPOSTA,Nunes Marques,2026-09-09
1,06017671020266000000,DIREITO DE RESPOSTA,Nunes Marques,2026-09-09
2,06017662520266000000,DIREITO DE RESPOSTA,Estela Aranha,2026-09-09


## Available columns

In [2]:
df.columns.tolist()

['processo',
 'classe',
 'assunto',
 'relator',
 'ementa',
 'decisao',
 'data_julgamento',
 'tipo_decisao',
 'sigla_classe',
 'sigla_uf',
 'municipio',
 'codigo_decisao',
 'url_inteiro_teor',
 'caminho_pdf']

## Previewing the ementa

`ementa`/`decisao` come from the backend as HTML fragments; the parser
strips the markup, so these are already plain text.

In [3]:
ementa = df["ementa"].dropna().iloc[0]
print(ementa[:300])

AGRAVO INTERNO. RECURSO ESPECIAL ELEITORAL. ELEIÇÕES 2024. REPRESENTAÇÃO. PEDIDO DE DIREITO DE RESPOSTA . DESCUMPRIMENTO. MULTA. RECURSO ESPECIAL INTERPOSTO APÓS O PRAZO DE 1 (UM) DIA ESTABELECIDO NO ART. 41 DA RES.–TSE Nº 23.608/2019. INTEMPESTIVIDADE. AUSÊNCIA DE IMPUGNAÇÃO ESPECÍFICA DOS FUNDAMEN


## Pagination cost model

Unlike other tribunals, each TSE page costs one full click + hCaptcha
handshake (roughly 1-2s) regardless of how many results it returns. The
UI only offers a few page sizes (`1/10/25/50/100/250`); `tamanho_pagina`
defaults to `250` (the largest) specifically to minimize the number of
pages needed for a full download. Pagination is also sequential: fetching
page `N` always clicks through the pages before it.

In [4]:
df_p2 = tse.cjsg("eleição", paginas=2, tamanho_pagina=10)
print(df_p2.shape)

(20, 14)


## Filtering by date

`data_julgamento_inicio/fim` (judgment date) and `data_publicacao_inicio/fim`
(publication date) are mutually exclusive in this version — the search UI
lets you combine both via a "+" button in the advanced dialog, but that
flow isn't implemented here yet (passing both pairs raises `ValueError`
immediately, before even launching a browser).

In [5]:
df_2022 = tse.cjsg(
    "eleição",
    paginas=1,
    tamanho_pagina=10,
    data_julgamento_inicio="01/01/2022",
    data_julgamento_fim="31/12/2022",
)
df_2022[["processo", "data_julgamento"]]

,processo,data_julgamento
0,06011118020226220000,2022-12-20
1,06006989420226210000,2022-12-20
2,06016077720226160000,2022-12-20
3,06001650220216200000,2022-12-19
4,06007588620206050128,2022-12-19
5,06009273920226000000,2022-12-19
6,06005726120226060000,2022-12-19
7,06005556220226190000,2022-12-19
8,06007542520226140000,2022-12-19
9,06004243420226030000,2022-12-19


## Downloading the inteiro teor (PDF)

`baixar_pdf=True` downloads the full-text PDF of every decision that has
one, via a plain (uncaptcha'd) `requests` GET — no browser needed for this
part. Files land in `download_path` (a temp dir by default) and the
DataFrame gets a `caminho_pdf` column.

In [ ]:
import tempfile

tse_pdf = jus.scraper("tse", download_path=tempfile.mkdtemp())
df_pdf = tse_pdf.cjsg("direito de resposta", paginas=1, tamanho_pagina=10, baixar_pdf=True)

com_pdf = df_pdf[df_pdf["caminho_pdf"].notnull()]
print(f"{len(com_pdf)} of {len(df_pdf)} decisions have a downloaded PDF")
com_pdf[["processo", "url_inteiro_teor", "caminho_pdf"]].head(3)

6 of 10 decisions have a downloaded PDF


,processo,url_inteiro_teor,caminho_pdf
4,06000695320246260278,https://sjur-servicos.tse.jus.br/sjur-servicos...,C:\Users\MELISS~1\AppData\Local\Temp\tmpgwcux7...
5,06000776520246260428,https://sjur-servicos.tse.jus.br/sjur-servicos...,C:\Users\MELISS~1\AppData\Local\Temp\tmpgwcux7...
6,06000417120266010000,https://sjur-servicos.tse.jus.br/sjur-servicos...,C:\Users\MELISS~1\AppData\Local\Temp\tmpgwcux7...


## Splitting download from parse

`cjsg` is a thin wrapper over `cjsg_download` (raw JSON per page) +
`cjsg_parse` (JSON → DataFrame). Splitting them is useful for caching the
raw pages before processing.

In [7]:
brutos = tse.cjsg_download("direito de resposta", paginas=1, tamanho_pagina=10)
print(f"{len(brutos)} page(s), {brutos[0]['totalRegistros']} total results")

df_again = tse.cjsg_parse(brutos)
df_again[["processo", "classe"]].head(3)

1 page(s), 1149 total results


,processo,classe
0,06017671020266000000,DIREITO DE RESPOSTA
1,06017689220266000000,DIREITO DE RESPOSTA
2,06017662520266000000,DIREITO DE RESPOSTA
